# RAG From Scratch: Overview

These notebooks walk through the process of building RAG app(s) from scratch.

They will build towards a broader understanding of the RAG langscape, as shown here:

![Advanced RAG](./Images/AdvancedRAG.png)

## Enviornment

`(1) Packages`

In [ ]:
!pip install -r requirements.txt

`(2) LangSmith`

https://smith.langchain.com/


`(3) API Keys`

- Langsmith
- GROQ
https://console.groq.com/docs/api-keys

## Set Environment Vars and API Keys
### 1. Loading Environment Variables

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

- **Purpose**: Load environment variables from a `.env` file.
- **Steps**:
  - `find_dotenv()`: Finds the path to the `.env` file.
  - `load_dotenv()`: Loads the environment variables from the `.env` file into the environment.

### 2. Setting Environment Variables


In [ ]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_PROJECT'] = 'advanced-rag-github'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['GROQ_API_KEY'] = os.getenv("GROQQ_API_KEY")

- **Purpose**: Set specific environment variables required for the application.
- **Steps**:
  - `os.environ['VAR_NAME'] = 'value'`: Sets the environment variable `VAR_NAME` to `value`.
  - `os.getenv("VAR_NAME")`: Retrieves the value of the environment variable `VAR_NAME`.




## Part 1: Overview
 
[RAG quickstart](https://python.langchain.com/docs/use_cases/question_answering/quickstart)

### 3. Importing Libraries and Initializing Components

In [ ]:
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

##1 - 0 - 1000 , 800 - 1800

# Split - Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
hf_embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)
vectorstore = FAISS.from_documents(documents=splits, 
                                    embedding=hf_embeddings)

retriever = vectorstore.as_retriever() # Dense Retrieval - Embeddings/Context based

#### RETRIEVAL and GENERATION ####

# Prompt
prompt = hub.pull("rlm/rag-prompt")

# LLM
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question
print(rag_chain.invoke("What is Task Decomposition?"))

This code snippet demonstrates a full workflow for creating a Retrieval-Augmented Generation (RAG) system using LangChain and related libraries. It involves loading web documents, splitting text into manageable chunks, embedding the text into a vector space, and finally using a language model to generate a response based on a user query. Below is a detailed line-by-line explanation of the code.

### 1. Import Statements

```python
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
```

- `import bs4`: This imports the BeautifulSoup4 (bs4) library, a popular library in Python used for parsing HTML and XML documents. It is typically used for web scraping purposes.
  
- `from langchain import hub`: Imports the `hub` module from LangChain, which is used to pull pre-built prompts from a repository or hub.
  
- `from langchain.text_splitter import RecursiveCharacterTextSplitter`: Imports the `RecursiveCharacterTextSplitter`, a utility in LangChain that is used to split long documents into smaller chunks of text based on character count. This helps in processing text efficiently.
  
- `from langchain_community.document_loaders import WebBaseLoader`: Imports `WebBaseLoader`, a class from the `langchain_community` package that loads documents from web pages.
  
- `from langchain_community.vectorstores import Chroma`: Imports the `Chroma` vector store from the `langchain_community` package, which stores document embeddings and allows for efficient similarity searches.
  
- `from langchain_core.output_parsers import StrOutputParser`: Imports `StrOutputParser`, a utility from LangChain's core module that parses the output of the language model into a string format.
  
- `from langchain_core.runnables import RunnablePassthrough`: Imports `RunnablePassthrough`, a utility that acts as a placeholder or a no-operation function in a processing chain.
  
- `from langchain_openai import ChatOpenAI, OpenAIEmbeddings`: Imports `ChatOpenAI` and `OpenAIEmbeddings` from `langchain_openai`, where `ChatOpenAI` is used to interact with OpenAI's GPT models, and `OpenAIEmbeddings` generates embeddings for text using OpenAI's model.

### 2. INDEXING

#### Load Documents

```python
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()
```

- `loader = WebBaseLoader(...)`: Instantiates a `WebBaseLoader` object that is used to load web documents from the provided URL.
  
- `web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",)`: Specifies the URL of the web page to load the document from. This URL points to a blog post.
  
- `bs_kwargs=dict(...)`: This dictionary contains arguments that are passed to BeautifulSoup for parsing the HTML content.
  
  - `parse_only=bs4.SoupStrainer(...)`: Specifies that only specific parts of the HTML should be parsed. In this case, only the parts of the HTML with the classes `post-content`, `post-title`, and `post-header` are parsed. This helps in narrowing down the content that is loaded and processed.
  
- `docs = loader.load()`: Calls the `load` method on the `loader` object to actually load and return the document(s) from the specified web page.

#### Split

```python
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
```

- `text_splitter = RecursiveCharacterTextSplitter(...)`: Creates an instance of the `RecursiveCharacterTextSplitter`. The `chunk_size=1000` argument specifies that each chunk of text should be approximately 1000 characters long, and `chunk_overlap=200` specifies that consecutive chunks should overlap by 200 characters.
  
- `splits = text_splitter.split_documents(docs)`: Splits the loaded document(s) into smaller chunks using the `text_splitter` object. The `splits` variable holds these chunks, which are later used for embedding.

#### Embed

```python
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()
```

- `vectorstore = Chroma.from_documents(...)`: Creates a `Chroma` vector store from the document chunks (`splits`). The `embedding=OpenAIEmbeddings()` argument indicates that OpenAI's embeddings will be used to convert the text chunks into vectors.
  
- `retriever = vectorstore.as_retriever()`: Converts the `vectorstore` into a retriever object. The retriever will be used later to fetch relevant chunks of text based on user queries.

### 3. RETRIEVAL and GENERATION

#### Prompt

```python
prompt = hub.pull("rlm/rag-prompt")
```

- `prompt = hub.pull("rlm/rag-prompt")`: Pulls a predefined RAG (Retrieval-Augmented Generation) prompt from the LangChain hub. This prompt is used as the basis for generating responses to user queries.

#### LLM

```python
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
```

- `llm = ChatOpenAI(...)`: Instantiates a `ChatOpenAI` object to interact with OpenAI's GPT-3.5-turbo model. The `temperature=0` argument sets the model's randomness to zero, making the responses more deterministic.

#### Post-processing

```python
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
```

- This defines a function `format_docs` that takes a list of document objects as input (`docs`) and concatenates their content (`doc.page_content`) into a single string, separated by double newlines.

#### Chain

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

- This part defines a processing chain using LangChain's operators (`|`).
  
- `{"context": retriever | format_docs, "question": RunnablePassthrough()}`: This dictionary sets up the input to the prompt. The `context` is created by retrieving relevant documents using `retriever` and then formatting them with `format_docs`. The `question` is simply passed through as-is using `RunnablePassthrough()`.
  
- `| prompt`: The input dictionary is then fed into the `prompt`, which likely formats the input as a query for the LLM.
  
- `| llm`: The formatted query is then sent to the language model (`llm`) to generate a response.
  
- `| StrOutputParser()`: Finally, the generated response is parsed into a string format using `StrOutputParser()`.

#### Question

```python
rag_chain.invoke("What is Task Decomposition?")
```

- `rag_chain.invoke(...)`: This method triggers the entire chain defined earlier. The input query, `"What is Task Decomposition?"`, is processed through the chain, resulting in a response generated by the LLM based on the retrieved and embedded documents.

### Summary

This code sets up a RAG pipeline using LangChain, where web content is fetched, split into chunks, embedded, and used in conjunction with a language model to answer specific queries. The entire process is modular, allowing for easy modification and extension, making it ideal for building sophisticated AI-driven systems.

## Part 2: Indexing

![Indexing](./Images/indexing.png)

In [ ]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

[Document Loaders](https://python.langchain.com/docs/integrations/document_loaders/)

In [ ]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

[Splitter](https://python.langchain.com/docs/modules/data_connection/document_transformers/recursive_text_splitter)

In [ ]:
# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)
splits

[Text embedding models](https://python.langchain.com/docs/integrations/text_embedding/openai)

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
hf_embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)
query_result = hf_embeddings.embed_query(question)
document_result = hf_embeddings.embed_query(document)
len(query_result)

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

In [ ]:
# Index
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents=splits, 
                                    embedding=hf_embeddings)

retriever = vectorstore.as_retriever()

In [ ]:
retriever

## Part 3: Retrieval

In [ ]:
docs = retriever.invoke("What is Task Decomposition?")
docs

In [ ]:
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

In [ ]:
# Chain
chain = prompt | llm

In [ ]:
chain.invoke({"context":docs,"question":"What is Task Decomposition?"})

[RAG chains](https://python.langchain.com/docs/expression_language/get_started#rag-search-example)

In [ ]:
from langchain import hub
prompt_hub_rag = hub.pull("rlm/rag-prompt")

In [ ]:
prompt_hub_rag

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_hub_rag
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")